[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/validation/labse-multilingual-smoketest/experiments/labse_english_hebrew_only.ipynb)


# LaBSE — Hebrew/English name matching

**Code cell 1:** Load the pretrained LaBSE model.

**Code cell 2:** Download the full, SHA-256-checked 200-record survey and tagged metadata from GitHub. Fields marked **N** contain English names or expressions. For every English expression in those fields, select the highest-scoring existing Hebrew candidate from the same field only if **cosine > α (initially 0.50)**. Review the scores and suggested replacements in a separate DataFrame; the original records are never overwritten.

The source workbook stays unchanged. The experiment does not train LaBSE or assert that a similarity score proves synonymy.



In [ ]:
# Cell 1 — download and load LaBSE once (PyTorch, no API key).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "sentence-transformers==5.1.2",
    ])

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")
print("LaBSE is loaded. Ready to compare English and Hebrew labels.")



In [ ]:
# Cell 2 — load the complete survey and its tagged metadata from GitHub.
# Your earlier notebook used a public HTTPS download with SHA-256 checking.
# No SSH key, Google Drive login, or manual upload is needed in Colab.
import hashlib
import io
import re
from urllib.request import urlopen

import numpy as np
import pandas as pd
from IPython.display import display


# Fixed snapshots of the latest 200-record workbook; the originals stay unchanged.
ROOT = "https://raw.githubusercontent.com/shulruferor-sudo/Advance-Data-Science/81ae343dd70fb4165a64fd498f5ea0b2c6c86a73/Datasets/cbs_occupation_survey_sample/"
SOURCES = {
    "survey": ("survey_200_records_current.csv", "c4928eedc97552f080968f816e1166e4ee4af1cda1aef42aab041d606a5b8625"),
    "metadata": ("survey_metadata_he_en_names.csv", "bf2d5034b777007bcdad5c545f70e90720dc220647fc5d0f1487740733d78c3e"),
}


def read_source(filename, expected_sha):
    # Download the exact CSV and verify its bytes before analyzing it.
    with urlopen(ROOT + filename, timeout=60) as response:
        content = response.read()
    assert hashlib.sha256(content).hexdigest() == expected_sha, filename
    return pd.read_csv(io.BytesIO(content), dtype="string", keep_default_na=False)


survey_df = read_source(*SOURCES["survey"])
metadata_df = read_source(*SOURCES["metadata"])

# N is the metadata marker for fields with Hebrew/English name-like text.
# Select columns by that marker, not by hardcoded column names.
name_columns = metadata_df.loc[
    metadata_df["HE_EN_NAMES"].eq("N"), "משתנה"
].tolist()

assert survey_df.shape == (200, 18)
assert survey_df["ID"].is_unique
assert all(column in survey_df.columns for column in name_columns)
print("Survey:", survey_df.shape, "| bilingual text fields:", name_columns)
display(metadata_df[["משתנה", "HE_EN_NAMES"]])

# Display the complete source dataframe, including all 200 records and 18 columns.
with pd.option_context("display.max_rows", 220, "display.max_columns", 18):
    display(survey_df)


# Alpha is the user-selected cosine threshold. It can be tuned later.
# A Latin expression may be a whole cell or part of a Hebrew sentence.
ALPHA = 0.50
LATIN = re.compile(r"[A-Za-z]+(?:[&./'-][A-Za-z]+)*(?:\s+[A-Za-z]+(?:[&./'-][A-Za-z]+)*)*")

# Preserve all source values. Write tentative translations to a COPY only.
proposed_df = survey_df.copy()
audit = []
changed_cells = []

for column in name_columns:
    values = survey_df[column].astype(str).tolist()

    # Extract every distinct English expression (case-insensitive detection).
    # Candidates must be Hebrew-only values already present in this column.
    english = sorted({word for value in values for word in LATIN.findall(value)})
    hebrew = sorted({value.strip() for value in values
                     if re.search(r"[\u0590-\u05FF]", value)
                     and not LATIN.search(value) and value.strip()})
    if not english or not hebrew:
        print(column, "— no English expressions or Hebrew-only candidates")
        continue

    # LaBSE produces normalized vectors; their dot product is cosine similarity.
    # Compare only English expressions against this column's Hebrew values.
    vectors = model.encode(english + hebrew, batch_size=8,
                           normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=False)
    cosine = vectors[:len(english)] @ vectors[len(english):].T
    replacements = {}

    for i, term in enumerate(english):
        j = int(np.argmax(cosine[i]))
        best_score = float(cosine[i, j])
        accepted = best_score > ALPHA
        if accepted:
            replacements[term] = hebrew[j]

        # Every distinct English expression gets one auditable top match.
        audit.append({"Column": column, "English": term,
                      "Best_Hebrew": hebrew[j], "Cosine": round(best_score, 4),
                      "Above_alpha": accepted})

    # Replace only the English substring, preserving any surrounding Hebrew.
    # Each original cell is inspected, and the outcome remains in a separate frame.
    for i, original in enumerate(values):
        if not LATIN.search(original):
            continue
        proposed = LATIN.sub(
            lambda match: replacements.get(match.group(), match.group()), original
        )
        proposed_df.at[i, column] = proposed
        changed_cells.append({"ID": survey_df.at[i, "ID"], "Column": column,
                              "Original": original, "Proposed": proposed,
                              "Changed": original != proposed})


# Show all highest-scoring candidates, plus every affected survey cell.
# A score above alpha is a proposal, not proof of a correct synonym.
audit_df = pd.DataFrame(audit).sort_values(["Column", "Cosine"], ascending=[True, False])
changed_df = pd.DataFrame(changed_cells)
print("\nTop candidate for each English expression:")
display(audit_df)
print("\nEvery cell containing English (including mixed Hebrew–English):")
display(changed_df)
print("\nProposed normalized dataframe (original survey_df remains unchanged):")
with pd.option_context("display.max_rows", 220, "display.max_columns", 18):
    display(proposed_df)

# Export review files. Do not update the source workbook automatically.
audit_df.to_csv("labse_name_candidate_scores.csv", index=False, encoding="utf-8-sig")
changed_df.to_csv("labse_english_cells_review.csv", index=False, encoding="utf-8-sig")
proposed_df.to_csv("survey_200_proposed_hebrew.csv", index=False, encoding="utf-8-sig")
print("COMPLETE — full dataset loaded; proposals ready for review; source unchanged.")

